# LeetCode 1417 — Reformat The String bằng Python

**Link đề:** https://leetcode.com/problems/reformat-the-string/
**Độ khó:** Easy · **Chủ đề:** String

```python
class Solution:
    def reformat(self, s: str) -> str:
        ...
```

## Mục tiêu học
1. Phân loại ký tự bằng `str.isdigit()` / `str.isalpha()` và tách chuỗi thành 2 nhóm.
2. Tìm **điều kiện có nghiệm**: `|#chữ − #số| <= 1` — và chứng minh vì sao.
3. **Đan xen** (interleave) hai danh sách: chỉ số chẵn/lẻ, `zip`, `itertools.zip_longest`, `str.join`.
4. Hiểu bài có **nhiều đáp án đúng** ⇒ test bằng **hàm kiểm tra tính hợp lệ** thay vì so chuỗi cố định.
5. Trace, Time/Space, lỗi hay gặp (bắt đầu bằng nhóm ít hơn), bài tập biến thể.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài

Cho chuỗi `s` gồm **chữ thường** và/hoặc **chữ số**. Tìm một **hoán vị** (sắp xếp lại các ký tự) sao cho **không có hai ký tự cạnh nhau cùng loại** (không chữ cạnh chữ, không số cạnh số).

Trả về chuỗi đó; nếu **không thể**, trả về chuỗi rỗng `""`.

| `s` | Một output hợp lệ | Giải thích |
|---|---|---|
| `"a0b1c2"` | `"0a1b2c"` | 3 chữ, 3 số → đan xen được; `"a0b1c2"`, `"0c2a1b"` cũng đúng |
| `"leetcode"` | `""` | chỉ có chữ |
| `"1229857369"` | `""` | chỉ có số |

**Constraints:** `1 <= s.length <= 500`, `s` chỉ gồm chữ thường và chữ số.

**Lưu ý quan trọng:** có **nhiều đáp án** — LeetCode chấp nhận bất kỳ đáp án hợp lệ nào. Ta sẽ viết hàm `is_valid_reformat(s, t)` để kiểm tra.

In [1]:
examples = [("a0b1c2", True), ("leetcode", False), ("1229857369", False)]
for s, possible in examples:
    print(f"s = {s!r:<14} -> {'có đáp án' if possible else 'trả về chuỗi rỗng'}")

s = 'a0b1c2'       -> có đáp án
s = 'leetcode'     -> trả về chuỗi rỗng
s = '1229857369'   -> trả về chuỗi rỗng


## 2. Nền tảng Python cần biết

### 2.1. `isdigit()` và `isalpha()`

- `"7".isdigit()` → `True`; `"a".isdigit()` → `False`.
- `"a".isalpha()` → `True`; `"7".isalpha()` → `False`.

In [2]:
for ch in "a0Z9":
    print(f"{ch!r}: isdigit={ch.isdigit()}, isalpha={ch.isalpha()}")

'a': isdigit=False, isalpha=True
'0': isdigit=True, isalpha=False
'Z': isdigit=False, isalpha=True
'9': isdigit=True, isalpha=False


### 2.2. Tách chuỗi thành 2 list bằng list comprehension

In [3]:
s = "a0b1c2"
letters = [ch for ch in s if ch.isalpha()]
digits = [ch for ch in s if ch.isdigit()]
print("letters =", letters)
print("digits  =", digits)

letters = ['a', 'b', 'c']
digits  = ['0', '1', '2']


### 2.3. `"".join(list)` — ghép list ký tự thành chuỗi

Chuỗi trong Python **bất biến** (immutable): không sửa được từng ký tự. Cách chuẩn: gom ký tự vào **list**, cuối cùng `"".join(list)`. Cộng chuỗi `+=` trong vòng lặp dễ tốn `O(n²)` ở trường hợp xấu.

In [4]:
parts = ["0", "a", "1", "b"]
print("".join(parts))
print("-".join(parts))
try:
    t = "abc"
    t[0] = "z"
except TypeError as e:
    print("TypeError:", e)

0a1b
0-a-1-b
TypeError: 'str' object does not support item assignment


### 2.4. `zip` và `itertools.zip_longest`

- `zip(A, B)` ghép cặp `(A[0], B[0]), (A[1], B[1]), ...` và **dừng ở list ngắn hơn**.
- `zip_longest(A, B, fillvalue="")` đi tới list **dài hơn**, chỗ thiếu điền `""`.

In [5]:
from itertools import zip_longest

A = ["a", "b", "c"]
B = ["1", "2"]
print("zip        :", list(zip(A, B)))
print("zip_longest:", list(zip_longest(A, B, fillvalue="")))

zip        : [('a', '1'), ('b', '2')]
zip_longest: [('a', '1'), ('b', '2'), ('c', '')]


### 2.5. Gán vào vị trí chẵn/lẻ bằng slicing có bước

`res[0::2] = X` gán `X` vào các vị trí `0, 2, 4, ...`; `res[1::2] = Y` gán vào `1, 3, 5, ...`. Độ dài phải khớp đúng số vị trí.

In [6]:
res = [None] * 5
res[0::2] = ["a", "b", "c"]      # vị trí 0, 2, 4
res[1::2] = ["1", "2"]           # vị trí 1, 3
print(res, "->", "".join(res))

['a', '1', 'b', '2', 'c'] -> a1b2c


## 3. Cách trực giác (baseline) — thử mọi hoán vị

"Thử mọi cách sắp xếp, cái nào hợp lệ thì trả về." Dùng `itertools.permutations`. Viết trước **hàm kiểm tra** vì ta sẽ dùng lại nhiều lần.

In [7]:
from itertools import permutations
from collections import Counter

def alternates(t: str) -> bool:
    # không có 2 ký tự cạnh nhau cùng loại
    return all(t[i].isdigit() != t[i + 1].isdigit() for i in range(len(t) - 1))

def is_valid_reformat(s: str, t: str) -> bool:
    # t là hoán vị của s và đan xen đúng
    return Counter(t) == Counter(s) and alternates(t)

print(alternates("0a1b2c"), alternates("ab12"))
print(is_valid_reformat("a0b1c2", "0a1b2c"), is_valid_reformat("a0b1c2", "0a1b2d"))

True False
True False


In [8]:
def reformat_bruteforce(s: str) -> str:
    for p in permutations(s):
        t = "".join(p)
        if alternates(t):
            return t
    return ""

for s, possible in examples[:1] + [("ab1", True), ("abc1", False)]:
    print(f"{s!r} -> {reformat_bruteforce(s)!r}")

'a0b1c2' -> 'a0b1c2'
'ab1' -> 'a1b'
'abc1' -> ''


### 3.1. Nút thắt

Số hoán vị của chuỗi dài `n` là `n!`. `10! = 3 628 800`, còn `500!` thì không tưởng. Với `"leetcode"` (8 ký tự, không có đáp án) baseline phải thử **hết** `8! = 40 320` hoán vị mới dám trả `""`.

⇒ Cần trả lời 2 câu hỏi **không cần thử**: (1) khi nào có đáp án? (2) nếu có, xây dựng trực tiếp thế nào?

In [9]:
from math import factorial
for n in [6, 8, 10, 20, 500]:
    print(f"n = {n:>3}: n! ≈ {float(factorial(n)):.3e}" if n < 171 else f"n = {n}: n! có {len(str(factorial(n)))} chữ số")

n =   6: n! ≈ 7.200e+02
n =   8: n! ≈ 4.032e+04
n =  10: n! ≈ 3.629e+06
n =  20: n! ≈ 2.433e+18
n = 500: n! có 1135 chữ số


## 4. Insight — đếm hai nhóm, rồi đan xen

Gọi `L` = số chữ, `D` = số chữ số. Một chuỗi đan xen có dạng `X Y X Y X ...`:
- Nếu độ dài chẵn: hai loại **bằng nhau**.
- Nếu độ dài lẻ: loại đứng đầu **nhiều hơn đúng 1**.

⇒ Có đáp án **khi và chỉ khi** `|L − D| <= 1`.

Xây dựng: **nhóm nhiều hơn đứng ở vị trí chẵn** (`0, 2, 4, ...`), nhóm còn lại ở vị trí lẻ. Nếu bằng nhau thì chọn nhóm nào trước cũng được.

In [10]:
for s in ["a0b1c2", "ab1", "a12", "abc1", "leetcode", "1"]:
    L = sum(ch.isalpha() for ch in s)
    D = len(s) - L
    print(f"{s!r:<12} L={L}, D={D}, |L-D|={abs(L - D)} -> {'có' if abs(L - D) <= 1 else 'không'} đáp án")

'a0b1c2'     L=3, D=3, |L-D|=0 -> có đáp án
'ab1'        L=2, D=1, |L-D|=1 -> có đáp án
'a12'        L=1, D=2, |L-D|=1 -> có đáp án
'abc1'       L=3, D=1, |L-D|=2 -> không đáp án
'leetcode'   L=8, D=0, |L-D|=8 -> không đáp án
'1'          L=0, D=1, |L-D|=1 -> có đáp án


`sum(ch.isalpha() for ch in s)` đếm số chữ: `True` được tính là `1`, `False` là `0`.

## 5. Lời giải chính — tách, kiểm tra, đan xen

**Đoạn 1 — tách:** `letters`, `digits`.

**Đoạn 2 — kiểm tra:** nếu `abs(len(letters) − len(digits)) > 1` → `""`.

**Đoạn 3 — chọn nhóm đi trước:** `a` = nhóm dài hơn (hoặc bằng), `b` = nhóm còn lại.

**Đoạn 4 — đan xen:** duyệt `i` từ 0: thêm `a[i]`, rồi thêm `b[i]` nếu còn.

In [11]:
def reformat_interleave(s: str) -> str:
    letters = [ch for ch in s if ch.isalpha()]          # Đoạn 1
    digits = [ch for ch in s if ch.isdigit()]
    if abs(len(letters) - len(digits)) > 1:             # Đoạn 2
        return ""
    a, b = (letters, digits) if len(letters) >= len(digits) else (digits, letters)   # Đoạn 3
    res = []
    for i in range(len(a)):                             # Đoạn 4
        res.append(a[i])
        if i < len(b):
            res.append(b[i])
    return "".join(res)

for s, possible in examples:
    t = reformat_interleave(s)
    print(f"{s!r:<14} -> {t!r:<10} hợp lệ? {is_valid_reformat(s, t) if t else (not possible)}")

'a0b1c2'       -> 'a0b1c2'   hợp lệ? True
'leetcode'     -> ''         hợp lệ? True
'1229857369'   -> ''         hợp lệ? True


`a, b = (X, Y) if điều_kiện else (Y, X)` là **biểu thức điều kiện** kết hợp **gán bộ** (tuple unpacking): gán 2 biến cùng lúc.

### 5.1. Bản dùng `zip_longest`

Mỗi cặp `(a[i], b[i])` ghép thành `a[i] + b[i]`; phần thiếu của `b` là `""`.

In [12]:
def reformat_zip(s: str) -> str:
    letters = [ch for ch in s if ch.isalpha()]
    digits = [ch for ch in s if ch.isdigit()]
    if abs(len(letters) - len(digits)) > 1:
        return ""
    if len(letters) < len(digits):
        letters, digits = digits, letters                # đổi chỗ để letters là nhóm dài hơn
    return "".join(x + y for x, y in zip_longest(letters, digits, fillvalue=""))

for s, _ in examples:
    print(f"{s!r} -> {reformat_zip(s)!r}")

'a0b1c2' -> 'a0b1c2'
'leetcode' -> ''
'1229857369' -> ''


## 6. Cách 3 — gán theo vị trí chẵn/lẻ bằng slicing

Tạo `res = [""] * n`. Nhóm dài hơn gán vào `res[0::2]` (có `ceil(n/2)` ô), nhóm ngắn hơn vào `res[1::2]` (có `floor(n/2)` ô). Điều kiện `|L − D| <= 1` đảm bảo độ dài khớp đúng.

In [13]:
def reformat_slots(s: str) -> str:
    letters = [ch for ch in s if ch.isalpha()]
    digits = [ch for ch in s if ch.isdigit()]
    if abs(len(letters) - len(digits)) > 1:
        return ""
    if len(letters) < len(digits):
        letters, digits = digits, letters
    res = [""] * len(s)
    res[0::2] = letters
    res[1::2] = digits
    return "".join(res)

for s, _ in examples + [("ab1", True), ("a12", True)]:
    print(f"{s!r} -> {reformat_slots(s)!r}")

'a0b1c2' -> 'a0b1c2'
'leetcode' -> ''
'1229857369' -> ''
'ab1' -> 'a1b'
'a12' -> '1a2'


## 7. Cách 4 — hai con trỏ trên list, không tạo 2 list phụ

Ý tưởng tiết kiệm bộ nhớ (theo tinh thần "in-place"): đếm `L`, `D`; xác định loại ở vị trí chẵn. Sau đó dùng hai con trỏ `i` (vị trí chẵn tiếp theo) và `j` (vị trí lẻ tiếp theo) để đặt từng ký tự vào mảng kết quả theo loại của nó — **một lượt duyệt**.

In [14]:
def reformat_two_pointers(s: str) -> str:
    L = sum(ch.isalpha() for ch in s)
    D = len(s) - L
    if abs(L - D) > 1:
        return ""
    even_is_alpha = L >= D                 # nhóm nhiều hơn đứng ở vị trí chẵn
    res = [""] * len(s)
    i, j = 0, 1                            # i: ô chẵn kế tiếp, j: ô lẻ kế tiếp
    for ch in s:
        if ch.isalpha() == even_is_alpha:
            res[i] = ch
            i += 2
        else:
            res[j] = ch
            j += 2
    return "".join(res)

for s, _ in examples + [("ab1", True), ("a12", True)]:
    print(f"{s!r} -> {reformat_two_pointers(s)!r}")

'a0b1c2' -> 'a0b1c2'
'leetcode' -> ''
'1229857369' -> ''
'ab1' -> 'a1b'
'a12' -> '1a2'


## 8. TRACE từng bước

Trace cách 4 với `s = "a12"` (`L = 1`, `D = 2` ⇒ số ở vị trí chẵn):

| ch | loại | về ô | i sau | j sau | res |
|---|---|---|---:|---:|---|
| `a` | chữ | lẻ `j=1` | 0 | 3 | `["", "a", ""]` |
| `1` | số | chẵn `i=0` | 2 | 3 | `["1", "a", ""]` |
| `2` | số | chẵn `i=2` | 4 | 3 | `["1", "a", "2"]` |

Kết quả `"1a2"` ✔. Điểm cần thấy: ký tự đầu là **chữ** nhưng không đứng đầu kết quả — vì nhóm **số** nhiều hơn.

In [15]:
def trace_reformat(s: str) -> str:
    L = sum(ch.isalpha() for ch in s)
    D = len(s) - L
    print(f"s={s!r}: L={L}, D={D}")
    if abs(L - D) > 1:
        print("  |L - D| > 1 -> trả về ''")
        return ""
    even_is_alpha = L >= D
    print(f"  vị trí chẵn dành cho: {'chữ' if even_is_alpha else 'số'}")
    res = [""] * len(s)
    i, j = 0, 1
    for ch in s:
        kind = "chữ" if ch.isalpha() else "số"
        if ch.isalpha() == even_is_alpha:
            res[i] = ch
            where = f"ô chẵn {i}"
            i += 2
        else:
            res[j] = ch
            where = f"ô lẻ {j}"
            j += 2
        print(f"  ch={ch!r} ({kind}) -> {where:<9} | i={i}, j={j}, res={res}")
    return "".join(res)

print("=>", repr(trace_reformat("a12")))
print()
print("=>", repr(trace_reformat("leetcode")))

s='a12': L=1, D=2
  vị trí chẵn dành cho: số
  ch='a' (chữ) -> ô lẻ 1    | i=0, j=3, res=['', 'a', '']
  ch='1' (số) -> ô chẵn 0  | i=2, j=3, res=['1', 'a', '']
  ch='2' (số) -> ô chẵn 2  | i=4, j=3, res=['1', 'a', '2']
=> '1a2'

s='leetcode': L=8, D=0
  |L - D| > 1 -> trả về ''
=> ''


Trace cách đan xen (mục 5) với `"a0b1c2"` — số bằng nhau nên chữ đi trước:

In [16]:
s = "a0b1c2"
letters = [ch for ch in s if ch.isalpha()]
digits = [ch for ch in s if ch.isdigit()]
res = []
for i in range(len(letters)):
    res.append(letters[i])
    if i < len(digits):
        res.append(digits[i])
    print(f"i={i}: thêm {letters[i]!r}" + (f" và {digits[i]!r}" if i < len(digits) else "") + f" -> {''.join(res)!r}")

i=0: thêm 'a' và '0' -> 'a0'
i=1: thêm 'b' và '1' -> 'a0b1'
i=2: thêm 'c' và '2' -> 'a0b1c2'


## 9. Code nộp LeetCode

In [17]:
class Solution:
    def reformat(self, s: str) -> str:
        letters = [ch for ch in s if ch.isalpha()]
        digits = [ch for ch in s if ch.isdigit()]
        if abs(len(letters) - len(digits)) > 1:
            return ""
        if len(letters) < len(digits):
            letters, digits = digits, letters
        res = []
        for i in range(len(letters)):
            res.append(letters[i])
            if i < len(digits):
                res.append(digits[i])
        return "".join(res)

for s, _ in examples:
    print(repr(s), "->", repr(Solution().reformat(s)))

'a0b1c2' -> 'a0b1c2'
'leetcode' -> ''
'1229857369' -> ''


### Bản slicing gọn

In [18]:
class SolutionSlots:
    def reformat(self, s: str) -> str:
        a = [c for c in s if c.isalpha()]
        d = [c for c in s if c.isdigit()]
        if abs(len(a) - len(d)) > 1:
            return ""
        if len(a) < len(d):
            a, d = d, a
        res = [""] * len(s)
        res[::2], res[1::2] = a, d
        return "".join(res)

print([SolutionSlots().reformat(s) for s, _ in examples])

['a0b1c2', '', '']


### Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Hoán vị | thử mọi cách sắp | `O(n! · n)` | `O(n)` | chỉ làm oracle |
| 2. Tách + đan xen | nhóm nhiều đi trước | `O(n)` | `O(n)` | rõ ràng nhất |
| 2b. `zip_longest` | ghép cặp | `O(n)` | `O(n)` | gọn |
| 3. Slicing chẵn/lẻ | `res[0::2]`, `res[1::2]` | `O(n)` | `O(n)` | rất ngắn |
| 4. Hai con trỏ | đặt trực tiếp theo loại | `O(n)` | `O(n)` (chỉ mảng kết quả) | một lượt |

## 10. Vì sao đúng?

**Điều kiện cần:** trong chuỗi đan xen độ dài `n`, vị trí chẵn có `⌈n/2⌉` ô, vị trí lẻ có `⌊n/2⌋` ô; mỗi loại chiếm trọn các ô cùng tính chẵn lẻ ⇒ hai số lượng chênh nhau tối đa 1. Vậy `|L − D| > 1` ⇒ không có đáp án ⇒ trả `""` là đúng.

**Điều kiện đủ (xây dựng):** giả sử `|L − D| <= 1`, gọi `a` là nhóm lớn hơn (`|a| = ⌈n/2⌉`), `b` nhóm nhỏ hơn (`|b| = ⌊n/2⌋`). Đặt `a` vào ô chẵn, `b` vào ô lẻ: hai ô kề nhau luôn khác tính chẵn lẻ ⇒ khác loại ✔. Mọi ký tự được dùng đúng một lần ⇒ là hoán vị ✔.

**Bất biến vòng đan xen:** sau bước `i`, `res` = `a[0] b[0] ... a[i] b[i]` (thiếu `b[i]` nếu hết `b`) — luôn đan xen, ký tự cuối là `a[i]` hoặc `b[i]`.

## 11. Time / Space

Gọi `n = len(s)`.

- Tách: 2 lượt `O(n)`; kiểm tra `O(1)`; đan xen `O(n)`; `join` `O(n)` ⇒ **`O(n)`**.
- Bộ nhớ: `letters`, `digits`, `res` ⇒ **`O(n)`** (không thể dưới `O(n)` vì phải trả về chuỗi mới).
- Hoán vị: `O(n! · n)` — không khả thi khi `n > 10`.

In [19]:
import random, string, time
random.seed(1)
big = "".join(random.choice(string.ascii_lowercase) for _ in range(250)) + "".join(random.choice(string.digits) for _ in range(250))
for f in [reformat_interleave, reformat_zip, reformat_slots, reformat_two_pointers]:
    t0 = time.perf_counter()
    for _ in range(500):
        t = f(big)
    print(f"{f.__name__:<22} hợp lệ={is_valid_reformat(big, t)} {1000 * (time.perf_counter() - t0):6.1f} ms / 500 lần")

reformat_interleave    hợp lệ=True   21.5 ms / 500 lần
reformat_zip           hợp lệ=True   21.6 ms / 500 lần
reformat_slots         hợp lệ=True   17.6 ms / 500 lần
reformat_two_pointers  hợp lệ=True   19.8 ms / 500 lần


`string.ascii_lowercase` = `"abc...z"`, `string.digits` = `"0123456789"`.

## 12. Bộ test

Vì có nhiều đáp án, test theo **tính chất**:
- Nếu có đáp án (`|L − D| <= 1`): kết quả phải là hoán vị và đan xen đúng.
- Nếu không: kết quả phải là `""`.
- Đối chiếu "có/không có đáp án" với vét cạn hoán vị trên chuỗi ngắn.

In [20]:
def check_reformat(name, s, t):
    L = sum(ch.isalpha() for ch in s)
    possible = abs(L - (len(s) - L)) <= 1
    if possible:
        assert is_valid_reformat(s, t), (name, s, t)
    else:
        assert t == "", (name, s, t)
    print(f"{name:<22} PASS -> {t!r}")

sol = Solution()
check_reformat("ví dụ 1", "a0b1c2", sol.reformat("a0b1c2"))
check_reformat("ví dụ 2", "leetcode", sol.reformat("leetcode"))
check_reformat("ví dụ 3", "1229857369", sol.reformat("1229857369"))
check_reformat("một chữ", "a", sol.reformat("a"))
check_reformat("một số", "7", sol.reformat("7"))
check_reformat("chữ nhiều hơn 1", "ab1", sol.reformat("ab1"))
check_reformat("số nhiều hơn 1", "a12", sol.reformat("a12"))
check_reformat("chênh 2", "abc1", sol.reformat("abc1"))
check_reformat("ký tự lặp", "aa11", sol.reformat("aa11"))

ví dụ 1                PASS -> 'a0b1c2'
ví dụ 2                PASS -> ''
ví dụ 3                PASS -> ''
một chữ                PASS -> 'a'
một số                 PASS -> '7'
chữ nhiều hơn 1        PASS -> 'a1b'
số nhiều hơn 1         PASS -> '1a2'
chênh 2                PASS -> ''
ký tự lặp              PASS -> 'a1a1'


In [21]:
methods = [reformat_interleave, reformat_zip, reformat_slots, reformat_two_pointers,
           Solution().reformat, SolutionSlots().reformat]
random.seed(1417)
for _ in range(400):
    s = "".join(random.choice("ab12") for _ in range(random.randint(1, 7)))
    brute = reformat_bruteforce(s)
    for f in methods:
        t = f(s)
        assert (t == "") == (brute == ""), (f, s, t, brute)
        if t:
            assert is_valid_reformat(s, t), (f, s, t)
for _ in range(500):
    s = "".join(random.choice("abcxyz0123") for _ in range(random.randint(1, 60)))
    for f in methods:
        check_ok = (f(s) == "") == (abs(2 * sum(c.isalpha() for c in s) - len(s)) > 1)
        assert check_ok and (f(s) == "" or is_valid_reformat(s, f(s))), (f, s)
print("Mọi cách khớp vét cạn (có/không đáp án) và luôn trả chuỗi hợp lệ")

Mọi cách khớp vét cạn (có/không đáp án) và luôn trả chuỗi hợp lệ


## 13. Lỗi người mới hay gặp

### Lỗi 1: Luôn bắt đầu bằng chữ cái

Khi số nhiều hơn chữ 1 đơn vị, bắt đầu bằng chữ sẽ "thừa" một số ở cuối, tạo hai số cạnh nhau (hoặc làm mất ký tự).

In [22]:
def wrong_letters_first(s):
    letters = [ch for ch in s if ch.isalpha()]
    digits = [ch for ch in s if ch.isdigit()]
    if abs(len(letters) - len(digits)) > 1:
        return ""
    return "".join(x + y for x, y in zip(letters, digits))   # SAI: zip cắt mất phần thừa

t = wrong_letters_first("a12")
print("sai:", repr(t), "hợp lệ?", is_valid_reformat("a12", t), "| đúng:", repr(reformat_interleave("a12")))

sai: 'a1' hợp lệ? False | đúng: '1a2'


### Lỗi 2: Dùng `zip` thay vì `zip_longest` → mất ký tự cuối

Như trên: `zip` dừng ở list ngắn hơn, ký tự thừa của nhóm dài bị **mất**. Luôn kiểm tra `len(kết_quả) == len(s)`.

### Lỗi 3: Điều kiện sai `L != D` → trả `""` cho trường hợp lệch 1 (vẫn có đáp án)

In [23]:
def wrong_condition(s):
    L = sum(ch.isalpha() for ch in s)
    if L != len(s) - L:                 # SAI: lệch 1 vẫn được
        return ""
    return reformat_interleave(s)

print("s='ab1': sai:", repr(wrong_condition("ab1")), "| đúng:", repr(reformat_interleave("ab1")))

s='ab1': sai: '' | đúng: 'a1b'


### Lỗi 4: Test bằng cách so với **một** chuỗi cố định

`assert sol.reformat("a0b1c2") == "0a1b2c"` có thể **fail** dù lời giải đúng (vì nó trả `"a0b1c2"`). Với bài nhiều đáp án, hãy test **tính chất** (`is_valid_reformat`).

In [24]:
out = Solution().reformat("a0b1c2")
print("output:", repr(out), "| so với '0a1b2c':", out == "0a1b2c", "| nhưng hợp lệ:", is_valid_reformat("a0b1c2", out))

output: 'a0b1c2' | so với '0a1b2c': False | nhưng hợp lệ: True


## 14. Mẹo nhận dạng pattern

- "Không có 2 phần tử kề nhau cùng loại" với **2 loại** ⇒ đếm hai loại, điều kiện `|A − B| <= 1`, loại nhiều hơn đứng đầu.
- Đan xen hai danh sách ⇒ `zip_longest` / slicing `[0::2]`, `[1::2]`.
- Bài **nhiều đáp án** ⇒ viết validator để test.
- Dạng tổng quát với **nhiều loại** (LeetCode 767 *Reorganize String*): điều kiện `max_count <= ⌈n/2⌉`, đặt loại nhiều nhất vào ô chẵn trước.

## 15. Biến thể

1. Bắt buộc đáp án **bắt đầu bằng số** nếu có thể (khi `L == D`).
2. Trả về đáp án **nhỏ nhất theo thứ tự từ điển** — sắp từng nhóm rồi đan xen, chọn nhóm đi trước sao cho chuỗi nhỏ hơn.
3. LeetCode 767 *Reorganize String*: không có 2 **ký tự giống nhau** kề nhau.
4. Đếm số cách sắp xếp hợp lệ (tổ hợp): nếu `L == D`: `2 · L! · D!` hoán vị vị trí (khi các ký tự khác nhau).

In [25]:
def reorganize_string(s: str) -> str:            # LeetCode 767
    cnt = Counter(s)
    n = len(s)
    ordered = sorted(cnt, key=lambda c: -cnt[c])  # ký tự nhiều nhất trước
    if cnt[ordered[0]] > (n + 1) // 2:
        return ""
    flat = [c for c in ordered for _ in range(cnt[c])]
    res = [""] * n
    res[0::2] = flat[:(n + 1) // 2]
    res[1::2] = flat[(n + 1) // 2:]
    return "".join(res)

def reformat_smallest(s: str) -> str:
    a = sorted(c for c in s if c.isalpha())
    d = sorted(c for c in s if c.isdigit())
    if abs(len(a) - len(d)) > 1:
        return ""
    candidates = []
    for first, second in [(a, d), (d, a)]:
        if len(first) >= len(second):
            res = [""] * len(s)
            res[0::2], res[1::2] = first, second
            candidates.append("".join(res))
    return min(candidates)

print(reorganize_string("aab"), repr(reorganize_string("aaab")))
print(reformat_smallest("c2a1b0"))

aba ''
0a1b2c


`sorted(cnt, key=lambda c: -cnt[c])`: `lambda` là **hàm vô danh** ngắn; ở đây sắp các ký tự theo số lần xuất hiện **giảm dần** (đổi dấu để sắp tăng thành giảm). Lý do `reorganize_string` đúng: ký tự nhiều nhất chiếm các ô chẵn đầu tiên nên không tự kề nhau; các ký tự khác được trải liên tục qua ô chẵn rồi ô lẻ.

## 16. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** Với `s = "ab123"`, có đáp án không? Nếu có, viết một đáp án.

**Bài 2 (đoán output):** `list(zip("ab", "123"))`, `"".join(["x", "1"])`, `"5".isalpha()`, `[0]*5` sau khi gán `[1,1,1]` vào `[0::2]`.

**Bài 3 (điền chỗ trống):** `exercise_reformat(s)` — tách, kiểm tra, đan xen.

**Bài 4 (tự viết):** `exercise_is_alternating(t)` — trả `True` nếu không có 2 ký tự kề nhau cùng loại.

**Bài 5 (biến thể):** `exercise_reformat_digit_first(s)` — giống bài chính, nhưng khi `L == D` thì **bắt buộc** bắt đầu bằng số.

**Bài 6 (mini challenge):** `exercise_reformat_smallest(s)` — đáp án nhỏ nhất theo thứ tự từ điển (hoặc `""`). Gợi ý: trong Python `"0" < "9" < "a"`.

In [26]:
def exercise_reformat(s: str) -> str:
    # TODO 1: tách letters, digits
    # TODO 2: |L - D| > 1 -> ""
    # TODO 3: nhóm dài hơn đi trước, đan xen
    pass

def exercise_is_alternating(t: str) -> bool:
    # TODO
    pass

# assert is_valid_reformat("a0b1c2", exercise_reformat("a0b1c2"))
# assert exercise_reformat("leetcode") == ""
# assert exercise_is_alternating("1a2b") and not exercise_is_alternating("ab1")

In [27]:
def exercise_reformat_digit_first(s: str) -> str:
    # TODO
    pass

def exercise_reformat_smallest(s: str) -> str:
    # TODO
    pass

# assert exercise_reformat_digit_first("a0b1") == "0a1b"
# assert exercise_reformat_smallest("c2a1b0") == "0a1b2c"

## 17. Gợi ý 3 mức

**Bài 1** — (1) `L = 2`, `D = 3`. (2) `|2 − 3| = 1` → có. (3) Số đi trước: `1a2b3`.

**Bài 3** — (1) Hai list comprehension. (2) `abs(...) > 1`. (3) `if len(letters) < len(digits): đổi chỗ`, rồi vòng `for i in range(len(letters))`.

**Bài 4** — (1) So từng cặp kề nhau `t[i]`, `t[i+1]`. (2) `t[i].isdigit() != t[i+1].isdigit()`. (3) `all(...)` trên `range(len(t) - 1)`.

**Bài 5** — (1) Khi `L == D`, cho `digits` đi trước. (2) Khi lệch 1, nhóm nhiều hơn **bắt buộc** đi trước. (3) Điều kiện đổi chỗ: `len(digits) >= len(letters)`.

**Bài 6** — (1) Sắp mỗi nhóm tăng dần. (2) Nếu `D >= L` thì số có thể đi trước — và số luôn nhỏ hơn chữ ⇒ ưu tiên số đi trước khi được phép. (3) Chỉ khi `L > D` mới bắt buộc chữ đi trước.

## 18. Đáp án tham khảo

**Bài 1:** có; ví dụ `"1a2b3"`.

**Bài 2:** `[('a', '1'), ('b', '2')]`, `"x1"`, `False`, `[1, 0, 1, 0, 1]`.

In [28]:
def answer_reformat(s):
    letters = [ch for ch in s if ch.isalpha()]
    digits = [ch for ch in s if ch.isdigit()]
    if abs(len(letters) - len(digits)) > 1:
        return ""
    if len(letters) < len(digits):
        letters, digits = digits, letters
    res = []
    for i in range(len(letters)):
        res.append(letters[i])
        if i < len(digits):
            res.append(digits[i])
    return "".join(res)

def answer_is_alternating(t):
    return all(t[i].isdigit() != t[i + 1].isdigit() for i in range(len(t) - 1))

def answer_reformat_digit_first(s):
    letters = [ch for ch in s if ch.isalpha()]
    digits = [ch for ch in s if ch.isdigit()]
    if abs(len(letters) - len(digits)) > 1:
        return ""
    first, second = (digits, letters) if len(digits) >= len(letters) else (letters, digits)
    res = [""] * len(s)
    res[0::2], res[1::2] = first, second
    return "".join(res)

def answer_reformat_smallest(s):
    letters = sorted(ch for ch in s if ch.isalpha())
    digits = sorted(ch for ch in s if ch.isdigit())
    if abs(len(letters) - len(digits)) > 1:
        return ""
    first, second = (digits, letters) if len(digits) >= len(letters) else (letters, digits)
    res = [""] * len(s)
    res[0::2], res[1::2] = first, second
    return "".join(res)

assert is_valid_reformat("ab123", "1a2b3") and answer_reformat("ab123") == "1a2b3"
z = [0] * 5; z[0::2] = [1, 1, 1]
assert list(zip("ab", "123")) == [("a", "1"), ("b", "2")] and "".join(["x", "1"]) == "x1"
assert "5".isalpha() is False and z == [1, 0, 1, 0, 1]
assert is_valid_reformat("a0b1c2", answer_reformat("a0b1c2")) and answer_reformat("leetcode") == ""
assert answer_is_alternating("1a2b") and not answer_is_alternating("ab1")
assert answer_reformat_digit_first("a0b1") == "0a1b"
assert answer_reformat_smallest("c2a1b0") == "0a1b2c"
random.seed(9)
for _ in range(300):
    s = "".join(random.choice("ba10") for _ in range(random.randint(1, 7)))
    sm = answer_reformat_smallest(s)
    valid = sorted({"".join(p) for p in permutations(s) if alternates("".join(p))})
    assert sm == (valid[0] if valid else ""), (s, sm)
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


Đáp án bài 6 được **kiểm tra với vét cạn** (sinh mọi hoán vị hợp lệ, lấy nhỏ nhất). Vì sao đúng: chuỗi đan xen bị **ép** loại theo vị trí; trong từng loại, đặt ký tự theo thứ tự tăng dần là tốt nhất; và khi được chọn loại đi đầu, số (`'0'..'9'`) luôn nhỏ hơn chữ.

## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def reformat(self, s: str) -> str:
        a = [c for c in s if c.isalpha()]
        d = [c for c in s if c.isdigit()]
        if abs(len(a) - len(d)) > 1:        # điều kiện có nghiệm
            return ""
        if len(a) < len(d):                 # nhóm nhiều hơn đứng ô chẵn
            a, d = d, a
        res = [""] * len(s)
        res[::2], res[1::2] = a, d          # đan xen
        return "".join(res)
```

**Checklist:**
- [ ] Điều kiện có nghiệm là `|L − D| <= 1` (lệch 1 vẫn được).
- [ ] Nhóm **nhiều hơn** đi trước.
- [ ] Không làm mất ký tự (`zip` cắt ngắn!) — độ dài kết quả = `len(s)`.
- [ ] Test bằng validator vì có nhiều đáp án đúng.